In [ ]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace, max as spark_max
from faker import Faker
from datetime import datetime
from datetime import timedelta
import random


import requests

# Get the list of files from the source
src_url = "https://api.github.com/repos/anirvandecodes/Azure-Databricks-Real-World-Healthcare-End-to-End-Project/contents/raw_data"

response = requests.get(src_url)
files = response.json()

src_files = [ file["name"] for file in files if file["name"] ] #.endswith(".csv")]
 

# Upload to the lakehouse
# Get existing file names from metadata table 
import os
import urllib.request as urllib
from urllib.parse import quote
from pyspark.sql import functions as F, Row


file_paths = '/lakehouse/default/Files/healthcare_data/'

"""# Load previously processed files
processed_df = spark.table('DE_Learning_LH.dbo.processed_files_table')

processed_files = set(
    processed_df.select("file_name").rdd.flatMap(lambda x: x).collect()
)"""


subfolders = ['diagnosis', 'hospital', 'patient', 'visit']
     

for file_name in src_files:
    """if file_name in processed_files:
        print(f"Skipping (already exists): {file_name}")
        continue """   
    
    # Determine subfolder dynamically based on prefix
    subfolder = None
    for folder in subfolders:
        if file_name.startswith(folder):
            subfolder = folder
            break

    if subfolder is None:
        print(f"No matching folder for: {file_name}, skipping...")
        continue

    url = f"{src_url}/{quote(file_name)}"  
    
    # ✅ Ensure subfolder exists
    target_dir = os.path.join(file_paths, subfolder)
    os.makedirs(target_dir, exist_ok=True)


    # Download only if not present
    urllib.urlretrieve(url, f"{target_dir}/{file_name}")

    # Updated rows in processed_files_table
    
    """new_files_df = spark.createDataFrame([
        Row(file_name=file_name, file_type = file_name.split('.')[-1]) 
    ])
    
    new_files_df = new_files_df.withColumn(
        "processed_time",
        F.current_timestamp()
    )


    print(new_files_df)
    new_files_df.write.mode("append").saveAsTable('DE_Learning_LH.dbo.processed_files_table')"""
       

    print(f"Uploaded: {file_name}")

